# Search Button Click 로그 SQL 생성
- event_name: search_button_click
- user_id: 1~100 (로그인) / null (비로그인)
- user_login_id: user0001~user0100 (로그인) / null (비로그인)
- client_uuid: 세션마다 고유 UUID

In [1]:
import random
import json
import uuid
from datetime import datetime, timedelta

In [2]:
# ───────────────────────────────────────────
# 설정값
# ───────────────────────────────────────────
SEARCH_KEYWORDS = [
    # 패션
    '나이키', '아디다스', '뉴발란스', '운동화', '신발', '슬리퍼', '샌들',
    '청바지', '반바지', '티셔츠', '후드티', '맨투맨', '가디건', '코트',
    '패딩', '점퍼', '자켓', '원피스', '블라우스', '스커트', '레깅스',
    '가방', '백팩', '토트백', '크로스백', '지갑', '벨트', '모자', '선글라스',
    # 뷰티
    '선크림', '비비크림', '파운데이션', '립스틱', '마스카라', '아이섀도우',
    '스킨케어', '에센스', '세럼', '수분크림', '폼클렌징', '미스트',
    '샴푸', '컨디셔너', '헤어에센스', '향수',
    # 식품
    '사과', '바나나', '딸기', '포도', '수박', '복숭아',
    '닭가슴살', '계란', '우유', '두부', '고구마', '아보카도',
    '커피', '녹차', '프로틴', '비타민', '홍삼',
    # 생활/가전
    '에어프라이어', '커피머신', '블렌더', '전기포트', '밥솥',
    '청소기', '공기청정기', '가습기', '선풍기', '전기장판',
    '이어폰', '무선이어폰', '블루투스스피커', '보조배터리', '충전기',
    '마우스', '키보드', '모니터', '웹캠', '태블릿',
    # 스포츠
    '요가매트', '덤벨', '폼롤러', '줄넘기', '헬스장갑',
    '등산화', '등산복', '자전거', '수영복', '테니스라켓',
    # 가구/인테리어
    '책상', '의자', '침대', '소파', '조명', '커튼', '러그',
    '수납함', '행거', '옷장', '화분', '캔들'
]

START_DATE     = datetime(2026, 1, 1, 0, 0, 0)
END_DATE       = datetime(2026, 6, 17, 23, 59, 59)
ROW_COUNT      = 500

# 비로그인 사용자 비율 (0.0 ~ 1.0)
ANONYMOUS_RATIO = 0.3

In [3]:
def random_datetime(start, end):
    delta = end - start
    random_seconds = random.randint(0, int(delta.total_seconds()))
    return start + timedelta(seconds=random_seconds)

def format_kst(dt):
    """event_timestamp 포맷 (KST +09:00)"""
    return dt.strftime('%Y-%m-%dT%H:%M:%S.') + f"{dt.microsecond // 1000:03d}+09:00"

def format_history_ts(dt):
    """history_timestamp 포맷 (마이크로초 포함)"""
    return dt.strftime('%Y-%m-%d %H:%M:%S.%f')

In [4]:
rows = []

for _ in range(ROW_COUNT):
    # history_timestamp 기준으로 먼저 뽑고, event_timestamp = history_ts + 1초
    history_ts = random_datetime(START_DATE, END_DATE)
    event_ts   = history_ts + timedelta(seconds=1)

    keyword     = random.choice(SEARCH_KEYWORDS)
    client_uuid = str(uuid.uuid4())

    # 비로그인 사용자 여부
    is_anonymous = random.random() < ANONYMOUS_RATIO

    if is_anonymous:
        user_id       = None
        user_login_id = None
    else:
        user_id       = random.randint(1, 100)
        user_login_id = f'user{user_id:04d}'

    json_log = json.dumps({
        'event_name':      'search_button_click',
        'searchKeyword':   keyword,
        'user_id':         user_id,
        'user_login_id':   user_login_id,
        'client_uuid':     client_uuid,
        'event_timestamp': format_kst(event_ts)
    }, ensure_ascii=False)

    rows.append((history_ts, json_log))

print(f'✅ {ROW_COUNT}개 search_button_click 로그 생성 완료')

✅ 500개 search_button_click 로그 생성 완료


In [5]:
# SQL 생성 및 저장
lines  = ['INSERT INTO first_save_history (history_timestamp, json_log) VALUES']
values = []

for history_ts, json_log in rows:
    ts_str  = format_history_ts(history_ts)
    escaped = json_log.replace("'", "''")
    values.append(f"  ('{ts_str}', '{escaped}')")

lines.append(',\n'.join(values) + ';')
sql = '\n'.join(lines)

with open('search_button_click_logs.sql', 'w', encoding='utf-8') as f:
    f.write(sql)

print(f'✅ {ROW_COUNT}개 search_button_click 로그 SQL 생성 완료 → search_button_click_logs.sql')

✅ 500개 search_button_click 로그 SQL 생성 완료 → search_button_click_logs.sql


In [6]:
# ── 미리보기 ──
print('=== SEARCH BUTTON CLICK SQL (앞 500자) ===')
print(sql[:500])

=== SEARCH BUTTON CLICK SQL (앞 500자) ===
INSERT INTO first_save_history (history_timestamp, json_log) VALUES
  ('2026-03-01 01:09:02.000000', '{"event_name": "search_button_click", "searchKeyword": "스킨케어", "user_id": null, "user_login_id": null, "client_uuid": "680ee18c-05ac-4835-a9e1-effdd3890f7d", "event_timestamp": "2026-03-01T01:09:03.000+09:00"}'),
  ('2026-03-02 17:53:34.000000', '{"event_name": "search_button_click", "searchKeyword": "점퍼", "user_id": 93, "user_login_id": "user0093", "client_uuid": "244bf0b3-a836-48da-b021-ff6f7d
